# Genome Firewall — live demo

Predict which antibiotics will **fail / work / can't-tell** from a bacterial genome —
with calibrated confidence, evidence, and honest abstention.

**Research prototype. All predictions must be confirmed with standard laboratory susceptibility testing.**

Models: https://huggingface.co/Darkroom4364/genome-firewall-ecoli · Hack-Nation 6th Global AI Hackathon

In [ ]:
!pip -q install skops huggingface_hub pandas matplotlib
import json, skops.io as sio, pandas as pd
from huggingface_hub import hf_hub_download

REPO = 'Darkroom4364/genome-firewall-ecoli'
bundle = json.load(open(hf_hub_download(REPO, 'demo_bundle.json')))
DRUGS = bundle['drugs']
models = {d: sio.load(hf_hub_download(REPO, f'models/{d}/model.skops'),
                      trusted=sio.get_untrusted_types(file=hf_hub_download(REPO, f'models/{d}/model.skops')))['model']
          for d in DRUGS}
print('loaded:', list(bundle['genomes']), '| drugs:', DRUGS)

## 1 · Three genomes, three stories

We picked three real E. coli genomes with known lab answers:
- **resistant** — carries the classic fluoroquinolone resistance mutations
- **susceptible** — clean, drug target verified intact
- **refusal** — a genome from an unseen lineage where honesty matters most

In [ ]:
def show_story(story):
    g = bundle['genomes'][story]
    print(f"=== {story.upper()} — genome {g['genome_id']} ===")
    hits = [h['symbol'] for h in g['evidence_hits'] if h['subtype'] in ('AMR','POINT','POINT_DISRUPT')]
    print(f"resistance signals found: {len(hits)}")
    print(' ', ', '.join(hits[:12]), ('…' if len(hits) > 12 else ''))
    print()

for s in ['resistant', 'susceptible', 'refusal']:
    show_story(s)

In [ ]:
def verdict_table(story):
    g = bundle['genomes'][story]
    cols = bundle['feature_columns']
    X = pd.DataFrame([{c: g['features'].get(c, 0) for c in cols}])
    rows = []
    for d in DRUGS:
        p = float(models[d].predict_proba(X)[0][1])
        lo, hi = g['predictions'][d]['band']
        verdict = 'likely to work' if p <= lo else ('likely to fail' if p >= hi else 'NO-CALL')
        lab = g['lab_labels'].get(bundle['drug_display'].get(d, d))
        truth = {1: 'RESISTANT', 0: 'susceptible'}.get(lab, '—')
        rows.append({'drug': d, 'P(fail)': round(p, 3), 'verdict': verdict, 'lab truth': truth})
    return pd.DataFrame(rows)

verdict_table('resistant')

In [ ]:
verdict_table('susceptible')

## 2 · The refusal — the whole point

This genome comes from a genetic lineage the model never saw in training.
Watch what the no-call layer does — and what a naive model would have said:

In [ ]:
df = verdict_table('refusal')
df['naive caller (p>0.5)'] = df['P(fail)'].apply(lambda p: 'FAIL' if p > 0.5 else 'work')
df

## 3 · The honest scoreboard

Evaluated on **held-out genetic groups** — bacterial lineages never seen in training
(skani ANI ≥ 99.5% single-linkage clusters; leakage audit passed).
This is the metric that matters; random splits inflate it.

In [ ]:
m = pd.DataFrame(bundle['metrics_heldout']).T[
    ['n', 'n_resistant', 'balanced_accuracy', 'recall_resistant',
     'recall_susceptible', 'auroc', 'brier', 'no_call_rate', 'accuracy_after_no_call']]
m.round(3)

## 4 · Try your own genome

Paste the contents of an AMRFinderPlus TSV below (one genome) — the notebook
maps its resistance elements onto the feature space and scores all 5 drugs.

In [ ]:
TSV_TEXT = """  # paste AMRFinderPlus TSV here (with header row)
"""

import io
def score_tsv(text):
    df = pd.read_csv(io.StringIO(text), sep='\t')
    cols = bundle['feature_columns']
    feats = {}
    for _, row in df.iterrows():
        sym = row.get('Element symbol')
        if sym in cols: feats[sym] = 1
        st = str(row.get('Subtype',''))
        if st.startswith('POINT') and f'POINT:{sym}' in cols: feats[f'POINT:{sym}'] = 1
        m = str(row.get('Method',''))
        if (m.startswith('PARTIAL') or m == 'INTERNAL_STOP') and f'DEG:{sym}' in cols: feats[f'DEG:{sym}'] = 1
    X = pd.DataFrame([{c: feats.get(c, 0) for c in cols}])
    out = []
    for d in DRUGS:
        p = float(models[d].predict_proba(X)[0][1])
        lo, hi = json.load(open(hf_hub_download(REPO, f'models/{d}/nocall_bands.json')))['band']
        out.append({'drug': d, 'P(fail)': round(p, 3),
                    'verdict': 'likely to work' if p <= lo else ('likely to fail' if p >= hi else 'NO-CALL')})
    return pd.DataFrame(out)

if TSV_TEXT.strip():
    display(score_tsv(TSV_TEXT))
else:
    print('paste a TSV above to score your own genome')

## 5 · Explore the full dataset — 1,434 genomes

The complete feature matrix + cleaned lab labels. Pick any genome (or a random one)
and compare the model's verdict with the lab's answer.

In [ ]:
fm = pd.read_csv(hf_hub_download(REPO, 'data/feature_matrix.csv'), dtype={'genome_id': str}).set_index('genome_id')
lab = pd.read_csv(hf_hub_download(REPO, 'data/labels_clean_ecoli.csv'), dtype={'genome_id': str})
cols = bundle['feature_columns']

def score_genome(gid):
    if gid not in fm.index:
        return f'{gid} not in dataset'
    X = fm.loc[[gid]]
    glab = lab[lab.genome_id == gid].set_index('antibiotic')['label'].to_dict()
    rows = []
    for d in DRUGS:
        p = float(models[d].predict_proba(X)[0][1])
        lo, hi = json.load(open(hf_hub_download(REPO, f'models/{d}/nocall_bands.json')))['band']
        v = 'likely to work' if p <= lo else ('likely to fail' if p >= hi else 'NO-CALL')
        truth = {1: 'RESISTANT', 0: 'susceptible'}.get(glab.get(bundle['drug_display'].get(d, d)), '—')
        rows.append({'drug': d, 'P(fail)': round(p, 3), 'verdict': v, 'lab truth': truth})
    return pd.DataFrame(rows)

# random genome with at least one label — change to any id, e.g. '562.100018'
gid = str(lab[lab.genome_id.isin(fm.index)].genome_id.sample(1, random_state=None).iloc[0])
print('genome:', gid)
score_genome(gid)

---
*Genome Firewall · strictly defensive: predicts and explains resistance that already exists,*
*never designs or modifies organisms. Every result requires confirmation by standard laboratory testing.*